<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 02. Escalado de Variables — Medir todo con la misma regla
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 05
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/05%20-%20Data%20Preparation/Para%20Dummies/02_Escalado_Caracteristicas_Data_Preparation_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

El escalado de variables es como **poner todas las medidas en las mismas unidades** antes de comparar. Aprenderás:

1. Por qué las escalas diferentes confunden a los algoritmos de ML.
2. **Normalización Min-Max** — comprimir todo entre 0 y 1.
3. **Estandarización Z-score** — centrar en 0 con desviación estándar 1.
4. **RobustScaler** — la versión resistente a outliers.
5. Cuándo usar cada técnica.

---
## 1. El problema de las escalas — Elefantes vs hormigas 🐘🐜

Imagina que quieres predecir el precio de una casa con dos variables:
- `área`: entre 50 y 500 m²
- `habitaciones`: entre 1 y 8

Si le dices a un algoritmo que ambas son igual de importantes, el área (que tiene valores 100 veces más grandes) **dominará completamente** el modelo — como si comparas el peso de un elefante con el de una hormiga.

El algoritmo pensará que 1 habitación más equivale a 1 m² más, cuando claramente no es así.

**La solución:** escalar ambas variables a la misma dimensión antes de entrenar.

> 💡 **¿Qué algoritmos necesitan escalado?**
> - ✅ KNN, SVM, Regresión Logística, Redes Neuronales, K-Means → **Sí necesitan**
> - ❌ Árbol de Decisión, Random Forest, XGBoost → **No necesitan** (no usan distancias)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# Dataset de ejemplo: características de casas
casas = pd.DataFrame({
    'area_m2':       [50, 80, 120, 200, 350, 500],
    'habitaciones':  [1,   2,   3,   4,   5,   8],
    'precio_millon': [80, 150, 200, 320, 480, 700]
})

print("🏠 Dataset original (escalas MUY diferentes):")
print(casas.to_string(index=False))
print(f"\nRango área: {casas['area_m2'].min()} - {casas['area_m2'].max()} m²")
print(f"Rango habitaciones: {casas['habitaciones'].min()} - {casas['habitaciones'].max()}")
print("\n⚠️ El área es 100x mayor que las habitaciones → el modelo le dará 100x más peso al área")

---
## 2. Normalización Min-Max — Todo entre 0 y 1 📏

La normalización **comprime** todos los valores al rango [0, 1]:
- El valor más pequeño se convierte en **0**.
- El valor más grande se convierte en **1**.
- Los demás quedan proporcionales entre sí.

**Fórmula:** `valor_nuevo = (valor - mínimo) / (máximo - mínimo)`

**Ejemplo cotidiano:** es como calificar películas del 0 al 10, no importa si el puntaje original era de 1 a 100.

In [ ]:
scaler_mm = MinMaxScaler()
casas_norm = pd.DataFrame(
    scaler_mm.fit_transform(casas[['area_m2', 'habitaciones']]),
    columns=['area_norm', 'habit_norm']
)

print("✨ Después de Normalización Min-Max (todo entre 0 y 1):")
print(pd.concat([casas[['area_m2', 'habitaciones']], casas_norm], axis=1).round(3).to_string(index=False))
print(f"\nRango área normalizada: {casas_norm['area_norm'].min():.1f} - {casas_norm['area_norm'].max():.1f}")
print(f"Rango habitaciones normalizadas: {casas_norm['habit_norm'].min():.1f} - {casas_norm['habit_norm'].max():.1f}")
print("\n✅ ¡Ahora ambas variables están en la misma escala!")

---
## 3. Estandarización Z-score — Centrar en 0 📐

La estandarización **transforma** los datos para que tengan:
- **Media = 0** (el promedio queda en el centro)
- **Desviación estándar = 1** (las unidades pasan a ser "desviaciones estándar")

**Fórmula:** `valor_nuevo = (valor - media) / desviación_estándar`

**Ejemplo cotidiano:** es como convertir todas las notas del colegio a una escala donde el promedio de la clase siempre es 0. Un estudiante con Z=+2 está 2 desviaciones por encima del promedio, y uno con Z=-1 está 1 desviación por debajo.

**¿Cuándo usar Z-score en lugar de Min-Max?**
- Cuando hay **outliers** — el Min-Max los comprime mucho, el Z-score los maneja mejor.
- Cuando el algoritmo asume distribución normal (muchos algoritmos estadísticos).

In [ ]:
scaler_std = StandardScaler()
casas_std = pd.DataFrame(
    scaler_std.fit_transform(casas[['area_m2', 'habitaciones']]),
    columns=['area_std', 'habit_std']
)

print("📐 Después de Estandarización Z-score (media=0, std=1):")
print(pd.concat([casas[['area_m2', 'habitaciones']], casas_std], axis=1).round(3).to_string(index=False))
print(f"\nMedia área estandarizada: {casas_std['area_std'].mean():.10f} ≈ 0")
print(f"Std área estandarizada:  {casas_std['area_std'].std():.4f} ≈ 1")

---
## 4. Comparación visual de los 3 escaladores 📊

In [ ]:
# Datos con outlier para ver diferencia
np.random.seed(42)
datos_con_outlier = np.concatenate([np.random.normal(50, 10, 99), [200]])  # 99 valores normales + 1 outlier
X = datos_con_outlier.reshape(-1, 1)

escaladores = {
    'Original': X.flatten(),
    'Min-Max (0-1)': MinMaxScaler().fit_transform(X).flatten(),
    'Z-score (μ=0)': StandardScaler().fit_transform(X).flatten(),
    'RobustScaler': RobustScaler().fit_transform(X).flatten()
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colores = ['#94a3b8', '#f59e0b', '#6366f1', '#10b981']

for ax, (nombre, datos), color in zip(axes, escaladores.items(), colores):
    ax.hist(datos, bins=20, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(f'{nombre}\nMin={datos.min():.2f}, Max={datos.max():.2f}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Valor')

plt.suptitle('Efecto de cada escalador (datos con un outlier en 200)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Observa: Min-Max comprime TODOS los datos al rango [0,1]")
print("   pero el outlier (200) jala todo hacia un extremo.")
print("   RobustScaler es más resistente a ese efecto.")

---
## 5. ¿Cuál usar? — Guía de decisión rápida 🗺️

| Situación | Escalador recomendado |
|---|---|
| Datos sin outliers, redes neuronales | **MinMaxScaler** (0 a 1) |
| Datos con distribución normal, SVM, Regresión | **StandardScaler** (Z-score) |
| Datos con outliers significativos | **RobustScaler** (usa mediana e IQR) |
| Árboles de decisión, Random Forest, XGBoost | **No necesita escalado** |

> ⚠️ **Regla de oro:** Siempre ajusta el escalador **solo con los datos de entrenamiento** (`fit_transform` en train), luego aplícalo a los datos de prueba (`transform` en test). Nunca ajustes con todos los datos juntos — eso introduce fuga de información (*data leakage*).

> 🚀 **Siguiente paso:** Ve al cuaderno `03_Fechas_y_Datos_Inconsistentes_Data_Preparation_Dummies.ipynb` para aprender a limpiar fechas con formatos raros y errores de escritura.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>